# ================================
#               IMPORTS          
# ================================

In [2]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
import json
import pandas as pd
from datetime import datetime
import re
import time
from datasets import load_dataset, concatenate_datasets
import edsnlp
from dotenv import load_dotenv
load_dotenv()


d:\oc\projet_14\medical-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [1]:
import torch 
print(torch.cuda.is_available())

True


# ================================
#        DATASET FOR SFT          
# ================================

In [ ]:
print("📥 Téléchargement des datasets publics...")
english_sft_dataset = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
french_sft_dataset = load_dataset("ANR-MALADES/MediQAl", name="oeq", split="test")

In [ ]:
english_sft_filtered = english_sft_dataset.filter(
    lambda x: x['qtype'] in ['symptoms', 'treatment']
)

english_shuffle = english_sft_filtered.shuffle(seed=42).select(range(1500))
french_shuffle = french_sft_dataset.shuffle(seed=42).select(range(2000))

In [ ]:
def estimer_priorite(texte):
    """Estime grossièrement la priorité en cherchant des mots-clés dans le texte."""
    texte_min = texte.lower()
    
    mots_urgence_haute = [
        "trauma", "cyanose", "cyanosis", "incohérent", "détresse", "distress",
        "hémorragie", "hemorrhage", "inconscient", "unconscious", "étouffe", "choking",
        "poitrine", "chest", "fracture ouverte", "fémur", "femur", "bassin", "pelvis",
        "polytraumatisé", "dyspnée", "dyspnea", "tachycardie", "tachycardia", 
        "paralysie", "paralysis"
    ]    

    # NIVEAU 2 : URGENCE MODÉRÉE / VIGILANCE
    mots_urgence_moderee = [
        "fièvre", "fever", "fracture", "brûlure", "burn", "douleur aiguë", "acute pain",
        "vomissement", "vomit", "vertige", "dizzy", "hypertension", "hta", 
        "high blood pressure", "diabète", "diabetes", "diabetic", "infection",
        "céphalée", "headache", "tremor"
    ]
    
    if any(mot in texte_min for mot in mots_urgence_haute):
        return 3
    elif any(mot in texte_min for mot in mots_urgence_moderee):
        return 2
    else:
        return 1

In [ ]:
# ===================================
#      EXTRACTION DES SYMPTOMES
# ===================================

with open("dictionnaire_symptomes.txt", "r", encoding="utf-8") as f:
    lignes = [line.strip() for line in f if line.strip() and not line.startswith("#")]

patterns_dict = {
    "SYMPTOME": lignes
}

print("🧠 Chargement du modèle EDS-NLP...")
nlp = edsnlp.blank("eds")
nlp.add_pipe("eds.sentences")     
nlp.add_pipe("eds.normalizer", config=dict(accents=True, lowercase=True))

nlp.add_pipe(
    "eds.matcher",
    config=dict(
        regex=patterns_dict, 
        attr="NORM"          
    ),
)

nlp.add_pipe("eds.negation")        

def extraire_symptomes_fr(texte):
    """Utilise EDS-NLP pour extraire intelligemment les symptômes non-niés."""
    if not texte or len(texte.strip()) < 4:
        return []

    doc = nlp(texte)
    symptomes_actifs = []
    
    symptomes_actifs = [ent.text.lower().strip() for ent in doc.ents if not ent._.negation]
    
    return list(set(symptomes_actifs))

In [ ]:
import spacy
from negspacy.negation import Negex

with open("dictionnaire_symptomes.txt", "r", encoding="utf-8") as f:
    symptomes_liste = [line.strip().lower() for line in f if line.strip() and not line.startswith("#")]

print("🧠 Chargement du modèle Spacy...")
nlp_en = spacy.load("en_core_web_sm")
ruler = nlp_en.add_pipe("entity_ruler", before="ner", config={"overwrite_ents": True})

patterns = []
for s in symptomes_liste:
    # On crée un pattern qui dit à spaCy : "Ceci est une expression régulière"
    patterns.append({
        "label": "SYMPTOME", 
        "pattern": [{"TEXT": {"REGEX": s}}]
    })
ruler.add_patterns(patterns)

nlp_en.add_pipe("negex", config={"ent_types": ["SYMPTOME"]})

def extraire_symptomes_en(texte):
    if not texte: return []
    doc = nlp_en(texte)
    
    return list(set([ent.text.lower() for ent in doc.ents if ent.label_ == "SYMPTOME" and not ent._.negex]))

In [ ]:
# test extraction
print(f"Test d'extraction : {extraire_symptomes_fr("Patiente diabétique avec une fistule purulente. Pas de fièvre.")}")

In [ ]:
print(f"Test d'extraction : {extraire_symptomes_en("Diabetic female patient with a purulent fistula. No fever.")}")

In [ ]:
date_du_jour = datetime.now().strftime("%Y-%m-%d")

def formater_mediqal_precis(ligne, index):
    """
    Adapte dynamiquement le format en fusionnant les colonnes 'clinical_case' 
    et 'question' si un cas patient existe réellement.
    """

    q_type = ligne.get("question_type") or ""
    case = ligne.get("clinical_case") or ""
    question = ligne.get("question") or ""
    reponse = ligne.get("answer") or ""
    
    case = case.strip()
    question = question.strip()
    symptomes_trouves = []
    
    if len(case) > 4:
        
        prompt_final = f" ### USER\n {case} {question}"
        priorite_calculee = estimer_priorite(prompt_final)
        symptomes_trouves = extraire_symptomes_fr(case)
        
        reponse_json ={
            "priorite": priorite_calculee,
            "analyse": reponse,
            "extraction_pour_le_medecin": {
                "symptomes": symptomes_trouves 
            }
        }
    else:
        prompt_final = f"### USER\n{question.strip()}"
        
        reponse_json = {
            "explication": reponse
        }
        
    reponse_ideale_string = f"### ANALYSE\n{json.dumps(reponse_json, ensure_ascii=False)}"

    return {
        "id_cas": f"CHSA-FR-{index:04d}",
        "prompt": prompt_final,
        "reponse_ideale": reponse_ideale_string, 
        "metadata": {
            "symptomes": symptomes_trouves ,
            "source": {"type_document": "ANR-MALADES/MediQAl", "date_creation": date_du_jour},
            "niveau_confiance": 5,
            "tag_origine": q_type 
        }
    }

In [ ]:
date_du_jour = datetime.now().strftime("%Y-%m-%d")

def formater_medquad_en(ligne, index):
    """Transforme une ligne du dataset anglais MedQuad vers le schéma CHSA."""

    q_type = ligne.get("qtype", "")
    question = ligne.get("Question", "")
    reponse = ligne.get("Answer", "")


    question = f"### USER\n{question.strip()}"
    reponse = reponse.strip()

    symptomes_trouves = extraire_symptomes_en(question)  

    reponse_json = {
        "explication": reponse
    }
    reponse_ideale_string = f"### ANALYSE\n{json.dumps(reponse_json, ensure_ascii=False)}"
    return {
        "id_cas": f"CHSA-EN-{index:04d}",
        "prompt": question,
        "reponse_ideale": reponse_ideale_string, 
        "metadata": {
            "symptomes": symptomes_trouves ,
            "source": {"type_document": "keivalya/MedQuad-MedicalQnADataset", "date_creation": date_du_jour},
            "niveau_confiance": 5,
            "tag_origine": q_type 
        }
    }

In [ ]:
french_chsa = french_shuffle.map(
    formater_mediqal_precis, 
    with_indices=True, 
    remove_columns=french_shuffle.column_names
)

In [ ]:
english_chsa = english_shuffle.map(
    formater_medquad_en, 
    with_indices=True, 
    remove_columns=english_shuffle.column_names
)

In [ ]:
from collections import Counter
def analyser_frequence_symptomes(dataset_chsa):
    """
    Parcourt le dataset final pour compter l'occurrence de chaque symptôme.
    """
    tous_les_symptomes = []
    
    for entree in dataset_chsa:
        # On récupère la liste des symptômes dans les métadonnées
        symptomes = entree.get("metadata", {}).get("symptomes", [])
        tous_les_symptomes.extend(symptomes)
    
    # Comptage des occurrences
    stats = Counter(tous_les_symptomes)
    
    # Transformation en DataFrame pour une lecture propre
    df_stats = pd.DataFrame(stats.items(), columns=['Symptôme', 'Occurrences'])
    return df_stats.sort_values(by='Occurrences', ascending=False)

In [ ]:
print("📊 Analyse des symptômes en cours...")

df_fr = analyser_frequence_symptomes(french_chsa)
df_en = analyser_frequence_symptomes(english_chsa)

print("\n🇫🇷 Top 10 Symptômes Français :")
print(df_fr.head(15).to_string(index=False))

print("\n🇺🇸 Top 10 Symptômes Anglais :")
print(df_en.head(15).to_string(index=False))

In [ ]:
print("\n✅ Exemple de la première ligne française formatée :")
print(json.dumps(french_chsa[0], indent=2, ensure_ascii=False))

In [ ]:
print("\n✅ Exemple de la première ligne anglaise formatée :")
print(json.dumps(english_chsa[0], indent=2, ensure_ascii=False))

# =======================================================
#         Création d'un dataset via mistral 
# ========================================================

In [ ]:
from mistralai.client import Mistral
from api.services.db_service import GSheetsDB

In [ ]:
# Initialiser le client Mistral
api_key = os.environ.get("MISTRAL_API_KEY")
client = Mistral(api_key=api_key)

# Initialiser la base de données 
db = GSheetsDB('credentials.json', 'CHSA_Triage_Dataset')

In [ ]:
def generate_batch_triage(taille_du_lot=5, start_index=0, langue="française", exemple=""):
    """
    Demande à Mistral de générer des données brutes, 
    puis les formate selon le standard CHSA (CAS CLINIQUE / REPONSE JSON).
    """
    
    prompt = f"""
        Tu es un médecin urgentiste expert. Génère {taille_du_lot} cas cliniques réalistes et créatifs en langue {langue} uniquement.
        Varie les profils (âge, sexe, symptômes, gravité).

        RÈGLES DE DIALOGUE :
        1. Chaque 'description' doit COMMENCER par la balise ### USER\\n\\n.
        2. Si le motif est flou, 'analyse_medicale' doit commencer par ### ASSISTANT\\n\\n pour poser une question.
        3. Si le motif est clair ou urgent, 'analyse_medicale' doit commencer par ### ANALYSE\\n\\n suivi du verdict.
        4. Pour les types 'Court', 'Questions' et 'Long', la 'description' doit contenir l'historique des échanges (alternance ### USER et ### ASSISTANT).

        TYPES À GÉNÉRER :
        Génère au moins un cas pour chaque type suivant : (Direct, first_question, Court, Questions, Long).

        STRUCTURE DE SORTIE :
        Tu DOIS répondre UNIQUEMENT par un objet JSON respectant exactement cette structure :
        {exemple}
        """

    try:
        response = client.chat.complete(
            model="mistral-small-latest",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"}, 
            temperature=0.8
        )
        
        raw_content = response.choices[0].message.content
        donnees = json.loads(raw_content)
        cas_generes = donnees.get("cas_cliniques", [])
        
        batch_formate = []
        
        for i, cas in enumerate(cas_generes):
            index_actuel = start_index + i
            
            # 1. Préparation du prompt unifié avec tes balises
            prompt_final = f"{cas['description']}\n"
            
            # 2. Extraction des symptômes via ton outil EDS-NLP
            if langue == "anglaise":
                symptomes_extraits = extraire_symptomes_en(cas['description'])
            else :
                symptomes_extraits = extraire_symptomes_fr(cas['description'])
                
            # 3. Construction de la réponse idéale (JSON structuré)
            reponse_obj = {
                "priorite": cas['priorite'],
                "analyse": cas['analyse_medicale'],
                "extraction_pour_le_medecin": {
                    "symptomes": symptomes_extraits 
                }
            }
            reponse_ideale_string = json.dumps(reponse_obj, ensure_ascii=False)
            
            # 4. Assemblage final au format CHSA
            entree_chsa = {
                "id_cas": f"CHSA-SYNTH-{index_actuel:04d}",
                "prompt": prompt_final,
                "reponse_ideale": reponse_ideale_string, 
                "metadata": {
                    "symptomes": symptomes_extraits,
                    "source": {
                        "type_document": "Mistral-Synthetic-Generation", 
                        "date_creation": datetime.now().strftime("%Y-%m-%d")
                    },
                    "niveau_confiance": 4, 
                    "tag_origine": f"mistral_{langue}_{cas['type']}"
                }
            }
            batch_formate.append(entree_chsa)
            
        return batch_formate
        
    except Exception as e:
        print(f"⚠️ Erreur lors de la génération/formatage : {e}")
        return []

In [ ]:
english_exemple = """
{{
  "cas_cliniques": [
    {{
      "type": "Direct",
      "description": "### USER\\nMy face is drooping on one side and I can't lift my right arm.",
      "analyse_medicale": "### ANALYSE\\nPossible Stroke (CVA). Sudden neurological deficit requires immediate emergency response. Call 911 now.",
      "priorite": 3
    }},
    {{
      "type": "first_question",
      "description": "### USER\\nI have a persistent cough that won't go away.",
      "analyse_medicale": "### ASSISTANT\\n Do you have any fever or shortness of breath? Are you coughing up any phlegm or blood?",
      "priorite": 1
    }},
    {{
      "type": "Court",
      "description": "### USER\\nI burned my hand on a hot stove.\\n\\n### ASSISTANT\\nAre there any blisters forming? Is the skin white, red, or charred?\\n\\n### USER\\nIt's very red and painful, but no blisters yet.",
      "analyse_medicale": "### ANALYSE\\nFirst-degree burn. Run lukewarm water over it for 20 minutes. Do not apply ice or butter. Monitor for changes.",
      "priorite": 1
    }},
    {{
      "type": "Questions",
      "description": "### USER\\nMy back really hurts after lifting some boxes.\\n\\n### ASSISTANT\\nDoes the pain travel down your legs or cause numbness?\\n\\n### USER\\nYes, I feel a shooting pain down my left leg.",
      "analyse_medicale": "### ASSISTANT\\nAre you experiencing any loss of bladder or bowel control?",
      "priorite": 2
    }},
    {{
      "type": "Long",
      "description": "### USER\\nI have a severe headache.\\n\\n### ASSISTANT\\nDid it come on suddenly like a 'thunderclap'?\\n\\n### USER\\nNo, it started gradually over the last few hours.\\n\\n### ASSISTANT\\nDo you have any neck stiffness or light sensitivity?\\n\\n### USER\\nNo, but I feel nauseous and see flashing lights.",
      "analyse_medicale": "### ANALYSE\\nPossible Migraine with aura. Rest in a dark, quiet room. If pain persists or worsens, consult a physician.",
      "priorite": 2
    }}
  ]
}}
"""

french_exemple = """
{{
  "cas_cliniques": [
    {{
      "type": "Direct",
      "description": "### USER\\nJe ressens une douleur thoracique violente qui serre comme un étau, je transpire et j'ai du mal à respirer.",
      "analyse_medicale": "### ANALYSE\\nSuspicion de syndrome coronaire aigu (Infarctus). Risque vital engagé. Appelez immédiatement le 15 ou le 112.",
      "priorite": 3
    }},
    {{
      "type": "first_question",
      "description": "### USER\\nJ'ai très mal au ventre depuis deux heures.",
      "analyse_medicale": "### ASSISTANT\\nOù se situe précisément la douleur (en haut, en bas, à droite) ? Est-ce que vous avez de la fièvre ou des vomissements ?",
      "priorite": 2
    }},
    {{
      "type": "Court",
      "description": "### USER\\nJe me suis coupé avec un couteau.\\n\\n### ASSISTANT\\nEst-ce que la plaie est profonde ? Arrivez-vous à bouger vos doigts normalement ?\\n\\n### USER\\nC'est une coupure superficielle sur le pouce, je bouge tout correctement.",
      "analyse_medicale": "### ANALYSE\\nPlaie cutanée simple. Nettoyez à l'eau et au savon, désinfectez et surveillez l'absence de rougeur. Vérifiez votre rappel de vaccin antitétanique.",
      "priorite": 1
    }},
    {{
      "type": "Questions",
      "description": "### USER\\nJ'ai des vertiges quand je me lève.\\n\\n### ASSISTANT\\nAvez-vous l'impression que la pièce tourne ou que vous allez vous évanouir ?\\n\\n### USER\\nPlutôt que je vais m'évanouir, j'ai les yeux qui se brouillent.",
      "analyse_medicale": "### ASSISTANT\\nAvez-vous pris vos médicaments ce matin ? Est-ce que vous avez bu assez d'eau aujourd'hui ?",
      "priorite": 1
    }},
    {{
      "type": "Long",
      "description": "### USER\\nMon enfant a de la fièvre.\\n\\n### ASSISTANT\\nQuelle est sa température précise ?\\n\\n### USER\\nIl a 39.5°C.\\n\\n### ASSISTANT\\nPrésente-t-il des taches rouges sur la peau ou une raideur dans la nuque ?\\n\\n### USER\\nNon, aucune tache, mais il est très fatigué.",
      "analyse_medicale": "### ANALYSE\\nFièvre isolée chez l'enfant. Administrez du paracétamol selon le poids et surveillez l'apparition de signes de gravité sous 4h.",
      "priorite": 2
    }}
  ]
}}
"""

In [ ]:
nombre_total_voulu = 500
taille_par_lot = 10
cas_generes = 0

print(f"🚀 Début de la génération : Objectif {nombre_total_voulu} cas...")


while cas_generes < nombre_total_voulu:
    print(f"\n⏳ Demande d'un lot de {taille_par_lot} cas à Mistral...")
    liste_nouveaux_cas = generate_batch_triage(taille_par_lot, cas_generes ,"française" ,french_exemple)
    
    if not liste_nouveaux_cas:
        print("⚠️ Échec du lot, on réessaye...")
        time.sleep(2)
        continue
        
    for entree_chsa in liste_nouveaux_cas:
        if not isinstance(entree_chsa, dict):
            continue
        
        try:   
            id_cas = entree_chsa["id_cas"]     
            prompt = entree_chsa["prompt"]
            metadata = entree_chsa["metadata"]
            symptomes_str = ", ".join(metadata["symptomes"])
            reponse = entree_chsa["reponse_ideale"]
            metadata_json = json.dumps(metadata, ensure_ascii=False)
        
            db.add_interaction(id_cas, prompt, reponse, metadata_json, symptomes_str)
        
            cas_generes += 1
            print(f"✅ [{id_cas}] ajouté avec succès.")

        except KeyError as e:
            print(f"❌ Erreur de structure dans l'objet retourné : Manque la clé {e}")
        except Exception as e:
            print(f"❌ Erreur lors de l'insertion en DB : {e}")

        if cas_generes >= nombre_total_voulu:
            break
            
    print(f"📊 Progression : {cas_generes} / {nombre_total_voulu}")
    time.sleep(30)

print("\n🎉 Mission accomplie : Dataset enrichi !")

# ==========================================
#    LA FUSION FINALE DU DATASET
# ==========================================

In [ ]:
from datasets import Dataset, concatenate_datasets

In [ ]:
mistral_generation = db.get_all_interactions()
mistral_dataset = Dataset.from_list(mistral_generation)

In [ ]:
print("🧬 Fusion des datasets...")

# Si vous avez formaté votre dataset de triage, ajoutez-le dans la liste ci-dessous !
dataset_sft_final = concatenate_datasets([english_chsa, french_chsa,mistral_dataset])
dataset_sft_final = dataset_sft_final.shuffle(seed=42)

print(f"📊 Taille du Full Dataset : {len(dataset_sft_final)} exemples.")

# ==========================================
#                SAUVEGARDE
# ==========================================

print("💾 Sauvegarde sur le disque...")
os.makedirs("data", exist_ok=True)
dataset_sft_final.to_json("../data/chsa_sft_full_dataset.jsonl", force_ascii=False)
print("✅ Terminé avec succès ! Prêt pour le SFTTrainer.")

# ==========================================
#          ANALYSE EXPLORATOIRE
# ==========================================

In [ ]:
df = dataset_sft_final.to_pandas()

In [ ]:
df.info()

In [ ]:
print(f"Total de lignes : {len(df)}")

# Analyse de la longueur des messages (bon indicateur de complexité)
df['prompt_len'] = df['prompt'].str.len()
df['reponse_len'] = df['reponse_ideale'].str.len()

print("\n--- Statistiques de longueur ---")
print(df[['prompt_len', 'reponse_len']].describe())

In [ ]:
# 1. On trie par longueur de réponse décroissante
top_10_long = df.sort_values(by='reponse_len', ascending=False).head(20)

print("🏆 TOP 20 DES RÉPONSES LES PLUS LONGUES")
print("-" * 50)

for i, (idx, row) in enumerate(top_10_long.iterrows(), 1):
    tag = row['metadata'].get('tag_origine', 'Inconnu')
    longueur = row['reponse_len']
    
    print(f"{i}. [Index {idx}] | Longueur : {longueur} car. | Tag : {tag}")
    # On affiche un aperçu du début et de la fin pour comprendre la structure
    aperçu = row['reponse_ideale']
    print(f"   Début : {aperçu[:150]}...")
    if longueur > 300:
        print(f"   Fin   : ...{aperçu[-150:]}")
    print("-" * 50)

In [ ]:
def tronquer_et_nettoyer(row):
    text = row['reponse_ideale']
    tag = row['metadata'].get('tag_origine', '')
    
    # Si c'est de la FAQ et que c'est trop long
    if tag in ['treatment', 'symptoms', 'understanding']:
        # 1. On nettoie le JSON si présent (on ne garde que le contenu de "explication")
        if '{"explication":' in text:
            import re
            match = re.search(r'"explication":\s*"(.*?)"', text, re.DOTALL)
            if match:
                text = match.group(1)
        
        # 2. On change la balise
        text = text.replace('### ANALYSE', '### ASSISTANT')
        if '### ASSISTANT' not in text:
            text = f"### ASSISTANT\n{text}"
            
        # 3. On plafonne à 2500 caractères (environ 600 mots)
        if len(text) > 2500:
            text = text[:2500] + "... [Texte tronqué pour l'entraînement : voir documentation complète]."
            
    return text

# Application du nettoyage final
df['reponse_ideale'] = df.apply(tronquer_et_nettoyer, axis=1)
df['reponse_len'] = df['reponse_ideale'].str.len()

print(f"✅ Nettoyage terminé. Longueur max désormais : {df['reponse_len'].max()} caractères.")

In [ ]:
duplicates = df.duplicated(subset=['prompt']).sum()
print(f"\nNombre de prompts en double : {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates(subset=['prompt'])

In [ ]:
from collections import Counter

all_symptoms = []

# On parcourt la colonne metadata
for meta in df['metadata']:
    # On vérifie que c'est bien un dictionnaire et qu'il contient la clé 'symptomes'
    if isinstance(meta, dict) and 'symptomes' in meta:
        liste_s = meta['symptomes']
        # On vérifie que c'est une liste (parfois c'est un array numpy vide [])
        if hasattr(liste_s, '__iter__') and not isinstance(liste_s, str):
            all_symptoms.extend([str(s).lower() for s in liste_s])

symptom_counts = Counter(all_symptoms)

print(f"--- Top 10 des symptômes (depuis metadata) ---")
if not all_symptoms:
    print("⚠️ Aucun symptôme trouvé dans les dictionnaires metadata.")
else:
    for symp, count in symptom_counts.most_common(10):
        print(f"{symp}: {count}")

In [ ]:
print(f"Contenu brut d'une cellule metadata :\n{df['metadata'].iloc[0]}")

In [ ]:
import matplotlib.pyplot as plt

df['prompt_len'].hist(bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution de la longueur des Prompts')
plt.xlabel('Nombre de caractères')
plt.ylabel('Nombre de cas')
plt.show()

In [ ]:
# 1. On calcule la longueur de chaque prompt
df['longueur'] = df['prompt'].str.len()

# 2. On récupère l'index du plus court et du plus long
idx_court = df['longueur'].idxmin()
idx_long = df['longueur'].idxmax()

print(f"📏 Longueur moyenne : {df['longueur'].mean():.0f} caractères")
print("-" * 50)

# 3. Affichage du cas le plus COURT
print(f"✅ CAS LE PLUS COURT (Index {idx_court}, {df.loc[idx_court, 'longueur']} caractères) :")
print(f"Prompt : {df.loc[idx_court, 'prompt']}")
print(f"Réponse : {df.loc[idx_court, 'reponse_ideale']}")

print("-" * 50)

# 4. Affichage du cas le plus LONG
print(f"🚨 CAS LE PLUS LONG (Index {idx_long}, {df.loc[idx_long, 'longueur']} caractères) :")
print(f"Prompt : {df.loc[idx_long, 'prompt'][:500]}...") # On coupe l'affichage si c'est trop long

In [ ]:

# 1. Extraction propre du tag_origine
def extraire_tag(meta):
    if isinstance(meta, dict):
        return meta.get('tag_origine', 'Inconnu')
    return 'Inconnu'

df['tag_provisoire'] = df['metadata'].apply(extraire_tag)

# 2. Calcul des fréquences (triées par ordre décroissant)
stats_tags = df['tag_provisoire'].value_counts().sort_values(ascending=False)

# 3. Création du graphique
plt.figure(figsize=(10, 6))
stats_tags.plot(kind='bar', color='teal', edgecolor='black')

# Personnalisation
plt.title('Répartition des sources du Dataset (tag_origine)', fontsize=14)
plt.xlabel('Source (Tag)', fontsize=12)
plt.ylabel('Nombre de cas', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Affichage des valeurs au-dessus des barres
for i, v in enumerate(stats_tags):
    plt.text(i, v + (max(stats_tags)*0.01), str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# On cherche le cas précis
cas_raisonnement = df[df['metadata'].apply(lambda x: x.get('tag_origine') == 'Raisonnement')]

# Affichage complet du contenu
if not cas_raisonnement.empty:
    print("🔍 EXAMEN DU CAS 'RAISONNEMENT' :")
    print("-" * 30)
    print(f"ID : {cas_raisonnement.iloc[0].get('id_cas', 'N/A')}")
    print(f"PROMPT :\n{cas_raisonnement.iloc[0]['prompt']}")
    print(f"\nRÉPONSE :\n{cas_raisonnement.iloc[0]['reponse_ideale']}")
    print("-" * 30)
    print(f"METADATA : {cas_raisonnement.iloc[0]['metadata']}")
else:
    print("⚠️ Aucun cas avec le tag 'Raisonnement' n'a été trouvé.")

In [ ]:
def corriger_tag(meta):
    if isinstance(meta, dict) and meta.get('tag_origine') == 'Raisonnement':
        # On crée une copie pour ne pas modifier l'original par accident
        nouveau_meta = meta.copy()
        nouveau_meta['tag_origine'] = 'Reasoning'
        return nouveau_meta
    return meta

# Application de la correction
df['metadata'] = df['metadata'].apply(corriger_tag)

# Vérification
tags_apres = df['metadata'].apply(lambda x: x.get('tag_origine')).unique()
print(f"✅ Tags mis à jour : {tags_apres}")

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
import re

def harmoniser_format_reponse(row):
    # 1. Récupération sécurisée du tag
    meta = row['metadata']
    tag = meta.get('tag_origine', '') if isinstance(meta, dict) else ''
    
    reponse = str(row['reponse_ideale'])
    tags_faq = ['symptoms', 'understanding', 'treatment']
    
    if tag in tags_faq:
        # 2. Remplacement de la balise
        nouvelle_reponse = reponse.replace('### ANALYSE', '### ASSISTANT')
        if '### ASSISTANT' not in nouvelle_reponse:
            nouvelle_reponse = f"### ASSISTANT\n{nouvelle_reponse}"
            
        # 3. BONUS NETTOYAGE : On enlève la structure JSON {"explication": "..."}
        # Ça permet au chatbot de parler comme un humain et pas comme un codeur
        nouvelle_reponse = reponse = re.sub(r'\{.*"explication":\s*"(.*?)".*\}', r'\1', nouvelle_reponse, flags=re.DOTALL)
        nouvelle_reponse = nouvelle_reponse.replace('{', '').replace('}', '').replace('"', '')
        
        return nouvelle_reponse.strip()
    
    # Pour les mises en situation, on ne touche à rien
    return reponse

# Application de la modification
print("🔄 Harmonisation des balises en cours...")
df['reponse_ideale'] = df.apply(harmoniser_format_reponse, axis=1)

# Vérification 1 : FAQ (Sécurisée avec isinstance)
filtre_faq = df['metadata'].apply(lambda x: x.get('tag_origine') == 'symptoms' if isinstance(x, dict) else False)
if not df[filtre_faq].empty:
    print("\n✅ Vérification FAQ (symptoms) :")
    print(df[filtre_faq].iloc[0]['reponse_ideale'][:150])

print("-" * 30)

# Vérification 2 : Mise en situation (Sécurisée avec isinstance)
filtre_mi = df['metadata'].apply(lambda x: x.get('tag_origine') not in ['symptoms', 'understanding', 'treatment'] if isinstance(x, dict) else False)
if not df[filtre_mi].empty:
    print("✅ Vérification Mise en Situation :")
    print(df[filtre_mi].iloc[0]['reponse_ideale'][:150])

In [ ]:
# 1. On cible uniquement les mises en situation (on ignore la FAQ)
filtre_scenario = df['metadata'].apply(lambda x: x.get('tag_origine') not in ['symptoms', 'Understanding', 'treatment'] if isinstance(x, dict) else False)

# 2. On cherche les cas où la réponse est anormalement courte (< 150 caractères)
cas_paresseux = df[filtre_scenario & (df['reponse_len'] < 100)]

print(f"🚨 Nombre de cas 'paresseux' trouvés : {len(cas_paresseux)}")

In [ ]:
cas_paresseux.head()

In [ ]:

def recategoriser_logique_utilisateur(row):
    meta = row['metadata']
    tag = meta.get('tag_origine', '') if isinstance(meta, dict) else ''
    reponse = str(row['reponse_ideale'])
    
    # TON MASQUE : Tag 'Reasoning' ET contient "explication" ET ne contient pas "priorite"
    if tag == 'Reasoning' and '"explication"' in reponse and '"priorite"' not in reponse:
        
        # 1. Mise à jour du tag (avec la Majuscule !)
        nouveau_meta = meta.copy()
        nouveau_meta['tag_origine'] = 'Understanding'
        
        # 2. Extraction du texte de l'explication
        texte_propre = reponse
        match = re.search(r'"explication":\s*"(.*?)"', reponse, re.DOTALL)
        if match:
            texte_propre = match.group(1)
            
        # Nettoyage des résidus JSON
        texte_propre = texte_propre.replace('{', '').replace('}', '').replace('"', '').strip()
        
        # 3. On applique la balise de réponse simple
        nouvelle_reponse = f"### ASSISTANT\n{texte_propre}"
        
        return pd.Series({'nouvelle_reponse': nouvelle_reponse, 'nouveau_meta': nouveau_meta})
        
    # Si la condition n'est pas remplie, on renvoie les données telles quelles
    return pd.Series({'nouvelle_reponse': reponse, 'nouveau_meta': meta})

print("🔄 Application de ton masque de filtrage...")

# Application du script
resultats = df.apply(recategoriser_logique_utilisateur, axis=1)

# Mise à jour du DataFrame avec les nouvelles valeurs
df['reponse_ideale'] = resultats['nouvelle_reponse']
df['metadata'] = resultats['nouveau_meta']

# Vérification du succès
print("✅ Opération terminée.")
nouveaux_tags = df['metadata'].apply(lambda x: x.get('tag_origine')).unique()
print(f"Liste des tags actuels : {nouveaux_tags}")

In [ ]:
# 1. On fixe la limite maximale acceptable pour une réponse
limite_caracteres = 3000

# 2. On compte combien de cas dépassent cette limite (pour vérifier ton intuition)
cas_trop_longs = df[df['reponse_len'] > limite_caracteres]
print(f"🚨 Nombre de cas ultra-longs trouvés (> {limite_caracteres} car.) : {len(cas_trop_longs)}")

# 3. SUPPRESSION : On garde uniquement les cas en dessous de la limite
df_final = df[df['reponse_len'] <= limite_caracteres].copy()

# 4. Vérification finale
print("-" * 30)
print(f"✅ Nettoyage terminé !")
print(f"Lignes avant : {len(df)}")
print(f"Lignes après suppression : {len(df_final)}")
print(f"Nouvelle longueur MAX d'une réponse : {df_final['reponse_len'].max()} caractères.")

In [ ]:
df_final.describe()


In [ ]:
df_final.to_csv("../data/df_clean_final.csv", index=False, encoding='utf-8-sig')
print("✅ Fichier sauvegardé avec succès dans ../data/df_clean_final.csv")

In [ ]:
def export_to_jsonl(dataframe, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for _, row in dataframe.iterrows():
            # Construction de la structure ChatML
            structure = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": row['prompt']},
                    {"role": "assistant", "content": row['reponse_ideale']}
                ]
            }
            # Écriture d'une ligne JSON
            f.write(json.dumps(structure, ensure_ascii=False) + "\n")

In [ ]:
df_metadata = pd.read_csv("../data/df_clean_final.csv", encoding='utf-8')

In [ ]:
df_metadata.info()

In [ ]:
def format_for_qwen(row):
    """
    Transforme chaque ligne au format ChatML natif attendu par Qwen.
    """
    system_prompt = (
        "Tu es un médecin urgentiste et régulateur expert. "
        "Ton rôle est d'analyser rapidement des situations cliniques pour effectuer un triage précis "
        "(déterminer le niveau d'urgence et extraire les symptômes clés), "
        "ou de répondre avec rigueur et clarté aux questions médicales théoriques."
    )    
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": row['prompt']},
            {"role": "assistant", "content": row['reponse_ideale']}
        ],
        "metadata": row['metadata']
    }

print("⚙️ Préparation du format ChatML pour Qwen...")

# 1. Application du format sur ton DataFrame nettoyé (df_final)
dataset_qwen = df_metadata.apply(format_for_qwen, axis=1).tolist()

# 2. Nom du fichier de sortie
chemin_complet = "../data/dataset_sft_qwen_final.jsonl"

# 3. Écriture du fichier JSONL
with open(chemin_complet, "w", encoding="utf-8") as f:
    for entry in dataset_qwen:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✅ EXPORT RÉUSSI POUR QWEN ! Le fichier '{chemin_complet}' contient {len(dataset_qwen)} cas prêts pour l'entraînement.")

# ==========================================
#          train_test_split
# ==========================================

In [ ]:
from datasets import DatasetDict

hf_dataset = Dataset.from_list(dataset_qwen)

train_val_et_test = hf_dataset.train_test_split(test_size=0.1, seed=42)

dataset_90_pourcent = train_val_et_test["train"]
dataset_test_final = train_val_et_test["test"]

train_et_val = dataset_90_pourcent.train_test_split(test_size=0.1111, seed=42)

mon_dataset_officiel = DatasetDict({
    "train": train_et_val["train"],           # ~ 80% des données 
    "validation": train_et_val["test"],       # ~ 10% des données 
    "test": dataset_test_final                # ~ 10% des données 
})

print(mon_dataset_officiel)

mon_dataset_officiel.save_to_disk("../data/chsa_dataset_split")

# ================================
#        DATASET FOR DPO          
# ================================

In [ ]:
from datasets import load_dataset
print("📥 Téléchargement de UltraMedical-Preference depuis Hugging Face...")
# On télécharge la partie 'train' du dataset
dataset_brut = load_dataset(
    "json",
    data_files={
        "train": "hf://datasets/TsinghuaC3I/UltraMedical-Preference/data/train.json"
    },
    split="train",
)

print(f"📊 Taille originale du dataset : {len(dataset_brut)} exemples.")

dataset_poc = dataset_brut.shuffle(seed=42).select(range(2000))
print(f"✂️ Dataset réduit à {len(dataset_poc)} exemples pour le POC.")

In [ ]:
system_prompt = (
    "Tu es un assistant médical expert du CHSA (Centre Hospitalier Sud-Ardennes). "
    "Ton rôle est d'apporter une aide à la décision clinique avec précision, pédagogie et rigueur scientifique. "
    "Pour chaque cas : 1) Analyse les signes de gravité (Red Flags). "
    "2) Propose une orientation ou une explication claire. "
    "3) Reste concis et professionnel. "
    "Tes réponses commencent toujours par la balise ### ASSISTANT."
)
def formater_en_pour_dpo(ligne):
    """Extrait les listes d'UltraMedical et applique les balises ChatML."""
    
    # Dans UltraMedical, 'chosen' est une liste : [{'role': 'user', 'content': '...'}, {'role': 'assistant', 'content': '...'}]
    question_user = ligne["chosen"][0]["content"]
    reponse_choisie = ligne["chosen"][1]["content"]
    reponse_rejetee = ligne["rejected"][1]["content"]
    
    # 1. Le Prompt (S'arrête avant la réponse de l'assistant)
    prompt_pour_dpo = f"{system_prompt}\n\nUser: {question_user}"
    chosen = f"### ASSISTANT\n{reponse_choisie}"
    rejected = f"### ASSISTANT\n{reponse_rejetee}"
    
    return {
        "prompt": prompt_pour_dpo,
        "chosen": chosen,
        "rejected": rejected
    }

def formater_fr_pour_dpo(ligne):
    """
    Adapté pour le DataFrame FR nettoyé (colonnes de texte pur).
    Prépare le triplet pour le DPOTrainer.
    """
    # On récupère le texte directement des colonnes de ton df_final_dpo_clean
    question_user = ligne["prompt_fr"]
    reponse_choisie = ligne["chosen_fr"]
    reponse_rejetee = ligne["rejected_fr"]
    
    # 1. Le Prompt (Le contexte + la question)
    # On garde la structure : System -> User
    prompt_pour_dpo = f"{system_prompt}\n\nUser: {question_user}"
    
    # 2. Les réponses formatées avec la balise Assistant
    # On utilise '### ASSISTANT' pour que le modèle sache où commence sa réponse
    chosen = f"### ASSISTANT\n{reponse_choisie}"
    rejected = f"### ASSISTANT\n{reponse_rejetee}"
    
    return pd.Series({
        "prompt": prompt_pour_dpo,
        "chosen": chosen,
        "rejected": rejected
    })

In [ ]:
def traduire_ligne_medicale(prompt_en, chosen_en, rejected_en):
    """Envoie les 3 textes d'un coup à l'API pour traduction."""
    
    instruction = (
        "Tu es un traducteur médical expert. Traduis ces trois textes de l'anglais vers le français. "
        "Garde un ton professionnel et une terminologie médicale précise. "
        "Réponds uniquement au format JSON avec les clés: 'prompt', 'chosen', 'rejected'."
    )
    
    contenu_a_traduire = f"PROMPT: {prompt_en}\nCHOSEN: {chosen_en}\nREJECTED: {rejected_en}"
    
    try:
        response = client.chat.complete(
            model="mistral-small-latest",
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": contenu_a_traduire}
            ],
            response_format={"type": "json_object"},
            temperature=0.2 # On baisse la température pour plus de fidélité
        )
        
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"⚠️ Erreur API : {e}")
        return None

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
import random
import time

df_poc = dataset_poc.to_pandas()

df_a_traduire, df_anglais_pur = train_test_split(df_poc, test_size=0.5, random_state=42)

df_a_traduire['prompt'] = df_a_traduire['prompt'].astype(object)
df_a_traduire['chosen'] = df_a_traduire['chosen'].astype(object)
df_a_traduire['rejected'] = df_a_traduire['rejected'].astype(object)

print(f"🚀 Traduction de {len(df_a_traduire)} lignes en cours...")

# 3. Boucle de traduction
for i, row in df_a_traduire.iterrows():
    resultat = traduire_ligne_medicale(row['prompt'], row['chosen'], row['rejected'])
    
    if resultat:
        df_a_traduire.at[i, 'prompt'] = resultat.get('prompt')
        df_a_traduire.at[i, 'chosen'] = resultat.get('chosen')
        df_a_traduire.at[i, 'rejected'] = resultat.get('rejected')
    
    if i % 20 == 0:
        print(f"✅ Avancement : {i}/{len(df_poc)}")
        # Sauvegarde intermédiaire au cas où
        df_a_traduire.to_csv("checkpoint_traduction.csv", index=False)
    
    temps_attente = random.uniform(1.0, 2.0)
    time.sleep(temps_attente)

# 4. Finalisation
df_a_traduire.to_csv("../data/df_dpo_fr_final.csv", index=False, encoding="utf-8-sig")
print("✨ Dataset mixte (FR/EN) prêt !")

In [ ]:
df_final.head()

In [ ]:
import pandas as pd
import json

def diagnostiquer_ligne_robuste(valeur):
    # 1. Gestion des types complexes (Listes/Arrays)
    if isinstance(valeur, (list, pd.Series)):
        if len(valeur) == 0: return "liste_vide"
        return "format_liste_brute" # Probablement la partie anglaise [User, Assistant]

    # 2. Cas : Valeur réellement manquante (None ou NaN)
    if pd.isna(valeur) is True:
        return "manquant"
    
    # 3. Cas : Dictionnaire (Résultat API déjà parsé)
    if isinstance(valeur, dict):
        if 'assistant' in valeur or 'content' in valeur: return "dictionnaire_ok"
        if 'translation' in valeur: return "dictionnaire_traduction_complexe"
        return "dictionnaire_inconnu"
    
    # 4. Cas : Chaîne de caractères
    if isinstance(valeur, str):
        v = valeur.strip()
        if v.startswith('{') and v.endswith('}'): return "json_string_a_parser"
        if len(v) < 10: return "texte_trop_court"
        if "None" in v and len(v) < 20: return "valeur_nulle_string"
        return "texte_brut_ok"
    
    return "format_inconnu"

# Application
stats_chosen = df_a_traduire['chosen'].apply(diagnostiquer_ligne_robuste).value_counts()

print("📊 BILAN DE SANTÉ DU DATASET :")
print(stats_chosen)

In [ ]:
def nettoyer_contenu_medical(valeur):
    # 1. Gestion des LISTES (Format original ou certains retours API)
    if isinstance(valeur, list):
        try:
            # On cherche le message de l'assistant (généralement à l'index 1 ou le dernier)
            if len(valeur) >= 2:
                # Si l'élément est un dictionnaire, on prend 'content'
                msg = valeur[1]
                if isinstance(msg, dict):
                    return msg.get('content', "")
                return str(msg)
            elif len(valeur) == 1:
                msg = valeur[0]
                return msg.get('content', "") if isinstance(msg, dict) else str(msg)
        except:
            return None

    # 2. Gestion des DICTIONNAIRES (Retours API typiques)
    if isinstance(valeur, dict):
        # Cas A : Dictionnaire simple avec 'assistant' ou 'content'
        if 'assistant' in valeur: return valeur['assistant']
        if 'content' in valeur: return valeur['content']
        
        # Cas B : Format complexe avec 'translation' / 'answer'
        if 'translation' in valeur:
            trans = valeur['translation']
            # Sécurité : 'translation' peut être une liste ou un dict
            fr_text = trans.get('fr', '') if isinstance(trans, dict) else ""
            
            # Récupération de la réponse
            ans = valeur.get('answer', {})
            ans_text = ans.get('fr', '') if isinstance(ans, dict) else ""
            
            return ans_text if len(str(ans_text)) > len(str(fr_text)) else fr_text
            
    # 3. Gestion du TEXTE BRUT
    if isinstance(valeur, str) and len(valeur) > 15:
        return valeur
        
    return None

# --- APPLICATION ---
print("🧹 Nettoyage des colonnes Chosen et Rejected...")
df_a_traduire['chosen_clean'] = df_a_traduire['chosen'].apply(nettoyer_contenu_medical)
df_a_traduire['rejected_clean'] = df_a_traduire['rejected'].apply(nettoyer_contenu_medical)

In [ ]:
df_a_traduire.head()

In [ ]:
def recuperer_prompt_fr(ligne):
    valeur = ligne['chosen'] # On fouille dans la donnée brute reçue de l'API
    
    # 1. Si c'est le format avec 'user' (le plus propre)
    if isinstance(valeur, dict) and 'user' in valeur:
        return valeur['user']
    
    # 2. Si c'est le format avec 'translation'
    if isinstance(valeur, dict) and 'translation' in valeur:
        trans = valeur['translation']
        if isinstance(trans, dict) and 'fr' in trans:
            # On nettoie si c'est une question à choix multiples
            return trans['fr'].split('\n\n')[0] 
            
    # 3. Si on ne trouve rien, on utilise la colonne prompt d'origine 
    # (Dernier recours : la question restera en anglais pour cette ligne)
    return ligne['prompt']

# Application de la récupération
df_a_traduire['prompt_clean'] = df_a_traduire.apply(recuperer_prompt_fr, axis=1)

In [ ]:
import pandas as pd
import numpy as np

def extraire_texte_universel(objet):
    """Extrait du texte de manière ultra-sécurisée."""
    # Test strict pour éviter le ValueError de Pandas/NumPy
    if isinstance(objet, (float, int)):
        if pd.isna(objet): return None
    if objet is None:
        return None
    
    # Si c'est une liste ou un array
    if isinstance(objet, (list, np.ndarray)):
        for item in objet:
            res = extraire_texte_universel(item)
            if res: return res
        return None

    # Si c'est un dictionnaire
    if isinstance(objet, dict):
        for cle in ['assistant', 'content', 'answer', 'fr', 'translation', 'user', 'question']:
            val = objet.get(cle)
            if val is not None:
                res = extraire_texte_universel(val)
                if res: return res
        return None

    # Si c'est du texte
    if isinstance(objet, str):
        texte = objet.strip()
        if len(texte) > 5:
            return texte
    
    return None

def nettoyage_chirurgical_final(ligne):
    # On récupère les objets bruts
    val_c = ligne.get('chosen')
    val_r = ligne.get('rejected')

    # Extraction du Prompt (Question)
    # Souvent dans le premier élément de la liste 'chosen'
    p = None
    if isinstance(val_c, list) and len(val_c) > 0:
        p = extraire_texte_universel(val_c[0])
    else:
        # Sinon on cherche partout dans l'objet chosen
        p = extraire_texte_universel(val_c)

    # Extraction de la réponse choisie (Assistant)
    c = None
    if isinstance(val_c, list) and len(val_c) > 1:
        c = extraire_texte_universel(val_c[1])
    else:
        c = extraire_texte_universel(val_c)

    # Extraction de la rejetée
    r = extraire_texte_universel(val_r)

    # Validation de sécurité
    # On vérifie que tout est du texte et que P != C
    if not isinstance(p, str) or not isinstance(c, str) or not isinstance(r, str):
        return pd.Series({'p': None, 'c': None, 'r': None})
    
    if p.strip() == c.strip():
        return pd.Series({'p': None, 'c': None, 'r': None})

    return pd.Series({'p': p, 'c': c, 'r': r})

# --- APPLICATION ---
print("🧹 Filtrage final (le vrai de vrai)...")
# On utilise result_type='expand' pour aider Pandas à créer les colonnes proprement
df_resultat = df_a_traduire.apply(nettoyage_chirurgical_final, axis=1)

# Nettoyage des lignes vides
df_final_fr = df_resultat.dropna(subset=['p', 'c', 'r']).copy()
df_final_fr.columns = ['prompt', 'chosen', 'rejected']

print(f"✅ Terminé ! {len(df_final_fr)} lignes extraites avec succès.")

In [ ]:
def extraction_stricte(valeur, est_prompt=False):
    # 1. Si c'est une liste (Format original ou Mistral a renvoyé une liste)
    if isinstance(valeur, list):
        if est_prompt and len(valeur) > 0:
            return valeur[0].get('content') # L'utilisateur
        if not est_prompt and len(valeur) > 1:
            return valeur[1].get('content') # L'assistant
        return None

    # 2. Si c'est un dictionnaire (Traduction Mistral)
    if isinstance(valeur, dict):
        if est_prompt:
            return valeur.get('user') or valeur.get('question')
        else:
            return valeur.get('assistant') or valeur.get('content') or valeur.get('answer')
    
    return None

def nettoyage_final_v3(ligne):
    # On force l'extraction par rôle
    p_fr = extraction_stricte(ligne['chosen'], est_prompt=True)
    c_fr = extraction_stricte(ligne['chosen'], est_prompt=False)
    r_fr = extraction_stricte(ligne['rejected'], est_prompt=False)

    # Si Mistral n'a pas traduit le USER dans 'chosen', 
    # on tente de voir s'il est dans 'rejected'
    if not p_fr:
        p_fr = extraction_stricte(ligne['rejected'], est_prompt=True)

    # VERIFICATION DE SÉCURITÉ CRITIQUE
    if not p_fr or not c_fr or not r_fr:
        return pd.Series({'p': None, 'c': None, 'r': None})
    
    # On vérifie que la réponse n'est pas la question
    if p_fr.strip() == c_fr.strip() or p_fr.strip() == r_fr.strip():
        return pd.Series({'p': None, 'c': None, 'r': None})

    return pd.Series({'p': p_fr, 'c': c_fr, 'r': r_fr})

# --- RE-LANCEMENT ---
df_final_fr = df_a_traduire.apply(nettoyage_final_v3, axis=1).dropna()

In [ ]:
import pandas as pd
import numpy as np

def extraire_reponse_rejected(valeur):
    """
    Extrait UNIQUEMENT le texte de la réponse (assistant) dans la colonne rejected.
    Utilise des tests d'identité stricts pour éviter les erreurs Pandas.
    """
    # Test de nullité sécurisé
    if valeur is None:
        return None
        
    # Gestion des types float (pour les NaN de Pandas)
    if isinstance(valeur, float):
        if np.isnan(valeur):
            return None

    # CAS A : C'est une LISTE (ou un array)
    if isinstance(valeur, (list, np.ndarray)):
        # On cherche l'élément qui a explicitement le rôle assistant
        for item in valeur:
            if isinstance(item, dict):
                if item.get('role') == 'assistant':
                    return item.get('content')
        
        # Si pas de rôle 'assistant' trouvé, on prend le contenu du dernier élément
        if len(valeur) > 0:
            dernier = valeur[-1]
            if isinstance(dernier, dict):
                return dernier.get('content')
            return str(dernier)

    # CAS B : C'est un DICTIONNAIRE
    if isinstance(valeur, dict):
        # On récupère les clés sans tester la vérité du dict
        res = valeur.get('content') or valeur.get('assistant') or valeur.get('answer')
        return str(res) if res is not None else None

    # CAS C : C'est déjà une chaîne
    if isinstance(valeur, str):
        return valeur.strip()

    return None

# --- APPLICATION ---
print("🧹 Nettoyage ciblé de la colonne Rejected...")
# On utilise axis=0 implicitement via apply sur la Series
df_a_traduire['rejected_clean'] = df_a_traduire['rejected'].apply(extraire_reponse_rejected)

In [ ]:
import re

def filtrage_qualite_rejected(texte):
    if texte is None or not isinstance(texte, str):
        return None
    
    # --- ÉTAPE A : Supprimer les QCM (A, B, C, D...) ---
    # On cherche des motifs comme "A.", "B.", "A)", "B)" en début de ligne ou isolés
    motif_qcm = r"(\n[A-D][\.\)])|(\b[A-D][\.\)]\s)"
    if re.search(motif_qcm, texte):
        return "QUALITE_INSUFFISANTE"

    # --- ÉTAPE B : Détecter les résidus JSON/Dictionnaires ---
    # Si le texte contient encore des marqueurs de structure technique
    marqueurs_techniques = ["{'", '{"', "[{", "'role':", "'content':", "'objectif':"]
    if any(m in texte for m in marqueurs_techniques):
        return "FORMAT_TECHNIQUE_RESIDUEL"

    # --- ÉTAPE C : Longueur minimale ---
    if len(texte.strip()) < 10:
        return "TROP_COURT"

    return texte.strip()

# Application du filtre sur la colonne déjà "nettoyée"
df_a_traduire['rejected_final'] = df_a_traduire['rejected_clean'].apply(filtrage_qualite_rejected)

# --- BILAN AVANT SUPPRESSION ---
stats_nettoyage = df_a_traduire['rejected_final'].value_counts()
print("📊 BILAN DU NETTOYAGE QUALITÉ (Rejected) :")
print(stats_nettoyage[stats_nettoyage.index.isin(["QUALITE_INSUFFISANTE", "FORMAT_TECHNIQUE_RESIDUEL", "TROP_COURT"])])

In [ ]:
# 1. On définit la liste des marqueurs à supprimer
exclure = ["QUALITE_INSUFFISANTE", "FORMAT_TECHNIQUE_RESIDUEL", "TROP_COURT"]

# 2. On filtre : on ne garde que ce qui n'est pas dans 'exclure' et qui n'est pas nul
df_rejected_nickel = df_a_traduire[
    (~df_a_traduire['rejected_final'].isin(exclure)) & 
    (df_a_traduire['rejected_final'].notna())
].copy()

# 3. On remplace l'ancienne colonne par la propre
df_rejected_nickel['rejected'] = df_rejected_nickel['rejected_final']

# 4. On supprime les colonnes de travail pour y voir clair
df_rejected_nickel = df_rejected_nickel.drop(columns=['rejected_clean', 'rejected_final'])

print(f"✨ Nettoyage Rejected terminé !")
print(f"Lignes restantes : {len(df_rejected_nickel)}")

In [ ]:
# 1. Voir ce qu'on a vraiment dans le DataFrame
print("📋 Colonnes disponibles :", df_rejected_nickel.columns.tolist())

# 2. Diagnostic sécurisé
# On affiche les 3 premières lignes avec ce qu'on est SÛR d'avoir
for i in range(3):
    ligne = df_rejected_nickel.iloc[i]
    print(f"\n" + "="*60)
    print(f"LIGNE {i}")
    print(f"="*60)
    
    # On affiche 'prompt' si il existe
    if 'prompt' in ligne:
        print(f"📝 PROMPT : {str(ligne['prompt'])[:100]}...")
    
    # On affiche 'chosen' (C'est là qu'on doit fouiller)
    if 'chosen' in ligne:
        print(f"💎 CHOSEN (Type: {type(ligne['chosen'])}) :")
        print(f"{str(ligne['chosen'])[:500]}...")
    
    # On affiche 'rejected' (Celle que tu as nettoyée)
    if 'rejected' in ligne:
        print(f"\n❌ REJECTED :")
        print(f"{str(ligne['rejected'])[:300]}...")

In [ ]:
import pandas as pd

def extraire_triplet_complet(ligne):
    ch = ligne['chosen']
    rej = ligne['rejected'] # Déjà nettoyé en texte pur par ton étape précédente
    
    p_fr, c_fr = None, None

    # CAS 1 : CHOSEN est une LISTE (Format complet : User + Assistant)
    if isinstance(ch, list):
        for item in ch:
            if not isinstance(item, dict): continue
            role = item.get('role', '').lower()
            content = item.get('content', '')
            
            if role == 'user':
                p_fr = content
            elif role == 'assistant':
                c_fr = content
        
        # Sécurité : si pas de rôles mais 2 éléments, on assume [Question, Réponse]
        if not p_fr and len(ch) >= 2:
            p_fr = ch[0].get('content') if isinstance(ch[0], dict) else None
            c_fr = ch[1].get('content') if isinstance(ch[1], dict) else None

    # CAS 2 : CHOSEN est un DICT (Format incomplet : Assistant seul)
    elif isinstance(ch, dict):
        # On ne récupère que la réponse, le prompt reste None
        c_fr = ch.get('content') or ch.get('assistant')

    # VALIDATION : On ne garde que si on a le PROMPT ET la RÉPONSE CHOSEN ET la RÉPONSE REJECTED
    if p_fr and c_fr and rej:
        # On vérifie que ce n'est pas du vide
        if len(str(p_fr)) > 10 and len(str(c_fr)) > 10:
            return pd.Series({'prompt_fr': p_fr, 'chosen_fr': c_fr, 'rejected_fr': rej})
    
    return pd.Series({'prompt_fr': None, 'chosen_fr': None, 'rejected_fr': None})

# --- EXECUTION ---
df_final_dpo = df_rejected_nickel.apply(extraire_triplet_complet, axis=1).dropna()

print(f"✨ Dataset DPO stabilisé !")
print(f"Nombre de triplets 100% français (Q&R) : {len(df_final_dpo)}")

In [ ]:
df_final_dpo.head()

In [ ]:
# Script d'inspection des 5 premières pépites
def inspecter_dataset(df, n=10):
    print(f"🔍 INSPECTION DES {n} PREMIERS TRIPLETS (FRANÇAIS)")
    print("="*80)
    
    for i in range(min(n, len(df))):
        ligne = df.iloc[i]
        print(f"\n📍 EXEMPLE n°{i+1}")
        print("-" * 30)
        
        # Affichage du Prompt
        print(f"❓ PROMPT (USER) :")
        print(f"{ligne['prompt_fr']}")
        print("-" * 15)
        
        # Affichage du Chosen
        print(f"✅ CHOSEN (ASSISTANT - BONNE RÉPONSE) :")
        print(f"{ligne['chosen_fr'][:500]}...") # On tronque à 500 car. pour la lisibilité
        print("-" * 15)
        
        # Affichage du Rejected
        print(f"❌ REJECTED (ASSISTANT - MAUVAISE RÉPONSE) :")
        print(f"{ligne['rejected_fr'][:500]}...")
        
        print("\n" + "="*80)

# Lancement de l'inspection
inspecter_dataset(df_final_dpo_clean)

In [ ]:
import pandas as pd

def inspecter_aleatoire(df, n=10):
    # .sample(n) mélange et tire n lignes au hasard
    df_sample = df.sample(n=min(n, len(df)))
    
    print(f"🎲 INSPECTION DE {len(df_sample)} TRIPLETS TIRÉS AU HASARD")
    print("="*80)
    
    for i in range(len(df_sample)):
        ligne = df_sample.iloc[i]
        print(f"\n📍 TIRAGE n°{i+1} (Index d'origine : {df_sample.index[i]})")
        print("-" * 30)
        
        print(f"❓ PROMPT (USER) :")
        print(f"{ligne['prompt_fr']}")
        print("-" * 15)
        
        print(f"✅ CHOSEN (BONNE RÉPONSE) :")
        print(f"{str(ligne['chosen_fr'])[:500]}...") 
        print("-" * 15)
        
        print(f"❌ REJECTED (MAUVAISE RÉPONSE) :")
        print(f"{str(ligne['rejected_fr'])[:500]}...")
        
        print("\n" + "="*80)

# Lancement du tirage aléatoire
inspecter_aleatoire(df_final_dpo_clean)

In [ ]:
def est_un_perroquet(ligne):
    p = str(ligne['prompt_fr']).strip().lower()
    c = str(ligne['chosen_fr']).strip().lower()
    
    # 1. Si la réponse est identique à la question (à 90% près)
    # On utilise une vérification simple de début de texte
    if c.startswith(p[:50]): 
        return True
    
    # 2. Si la réponse est trop courte par rapport à une question longue
    if len(c) < len(p) * 0.8 and len(p) > 100:
        return True
        
    return False

# Filtrage final
masque_perroquet = df_final_dpo.apply(est_un_perroquet, axis=1)
df_final_dpo_clean = df_final_dpo[~masque_perroquet].copy()

print(f"✂️ Lignes 'perroquets' supprimées : {masque_perroquet.sum()}")
print(f"💎 Triplets finaux ultra-qualitatifs : {len(df_final_dpo_clean)}")

In [ ]:
df_final_dpo_clean.head()

In [ ]:
df_en_20 = df_anglais_pur.sample(frac=0.20, random_state=42).copy()

In [ ]:
df_en_20.head()

In [ ]:
df_en_dpo = df_en_20.apply(formater_en_pour_dpo, axis=1).dropna()
df_fr_fr = df_final_dpo_clean.apply(formater_fr_pour_dpo, axis=1)

In [ ]:
print("⚙️ Application du formatage ChatML...")

df_dpo_final = pd.concat([df_fr_fr, df_en_dpo], axis=0)
df_final_clean = df_dpo_final.sample(frac=1, random_state=42).reset_index(drop=True)

df_final_clean.head()

In [ ]:
df_final_clean.info()

In [ ]:
import pandas as pd

# 1. On récupère les colonnes existantes pour voir le problème
print("Avant nettoyage :", df_final_clean.columns.tolist())

# 2. Si les données EN sont dans la colonne '0', on les "déplie"
# On crée un nouveau DataFrame propre
rows = []

for idx, ligne in df_final_clean.iterrows():
    # Si la ligne a des données dans la colonne '0' (format dict/Series)
    if 0 in ligne and pd.notna(ligne[0]):
        item = ligne[0]
        # On extrait selon si c'est un dict ou un objet
        rows.append({
            'prompt': item.get('prompt') if isinstance(item, dict) else getattr(item, 'prompt', None),
            'chosen': item.get('chosen') if isinstance(item, dict) else getattr(item, 'chosen', None),
            'rejected': item.get('rejected') if isinstance(item, dict) else getattr(item, 'rejected', None)
        })
    else:
        # Sinon on prend les colonnes FR classiques
        rows.append({
            'prompt': ligne.get('prompt'),
            'chosen': ligne.get('chosen'),
            'rejected': ligne.get('rejected')
        })

# 3. On reconstruit le DataFrame final
dataset_dpo_final = pd.DataFrame(rows)

# 4. Suppression des lignes vides éventuelles et reset d'index
dataset_dpo_final = dataset_dpo_final.dropna(subset=['prompt', 'chosen', 'rejected']).reset_index(drop=True)

# --- VÉRIFICATION FINALE ---
print("\nAprès nettoyage :", dataset_dpo_final.columns.tolist())
print(dataset_dpo_final.info())

In [ ]:

dataset_dpo_final.to_json("../data/dataset_dpo_qwen_final.jsonl",orient='records', lines=True,force_ascii=False)
print("Terminé avec succès !")

# ========================================
#           ANONYMISATION 
# ========================================

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# 1. Configurer le moteur NLP pour charger SpaCy et le modèle français
configuration = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "fr", "model_name": "fr_core_news_md"},
        {"lang_code": "en", "model_name": "en_core_web_md"},
    ],
}

# 2. Créer le fournisseur et le moteur
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()

# 3. Initialiser l'Analyseur et l'Anonymiseur
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, default_score_threshold=0.4)
anonymizer = AnonymizerEngine()

print("✅ Moteur Presidio initialisé sans PorterStemmer !")

In [ ]:
from presidio_analyzer import PatternRecognizer, Pattern

# --- 1. CONFIGURATION DES PATTERNS (REGEX) ---

# Français (FR)
date_fr_pattern = Pattern(name="date_fr", regex=r"(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})", score=0.95)
phone_fr_pattern = Pattern(name="phone_fr", regex=r"(?:(?:\+|00)33|0)\s*[1-9](?:[\s.-]*\d{2}){4}", score=0.95)

# Anglais (EN - US Address & Phone)
us_address_pattern = Pattern(name="us_address", regex=r"\d{1,5}\s\w+(\s\w+)?\s(Street|St|Avenue|Ave|Road|Rd|Terrace|Ter|Drive|Dr)", score=0.6)
us_phone_pattern = Pattern(name="us_phone", regex=r"(\d{3}-\d{3}-\d{4})|(\(\d{3}\)\s\d{3}-\d{4})|(\d{3}-\d{4})", score=0.7)

# --- 2. CRÉATION DES RECOGNIZERS ---

# Recognizers FR
date_fr_rec = PatternRecognizer(supported_entity="DATE_TIME", patterns=[date_fr_pattern], supported_language="fr")
phone_fr_rec = PatternRecognizer(supported_entity="PHONE_NUMBER", patterns=[phone_fr_pattern], supported_language="fr")

# Recognizers EN
address_en_rec = PatternRecognizer(supported_entity="LOCATION", patterns=[us_address_pattern], supported_language="en")
phone_en_rec = PatternRecognizer(supported_entity="PHONE_NUMBER", patterns=[us_phone_pattern], supported_language="en")

# --- 3. ENREGISTREMENT DANS L'ANALYZER ---

# On ajoute tout au registre
analyzer.registry.add_recognizer(date_fr_rec)
analyzer.registry.add_recognizer(phone_fr_rec)
analyzer.registry.add_recognizer(address_en_rec)
analyzer.registry.add_recognizer(phone_en_rec)

print(f"✅ Configuration terminée !")
print(f"🌍 Langues supportées par l'analyseur : {analyzer.supported_languages}")

In [ ]:
import json

# Pour le DPO
dataset_dpo_qwen_final = []
with open('../data/dataset_dpo_qwen_final.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        dataset_dpo_qwen_final.append(json.loads(line))

# Pour le SFT
dataset_sft_qwen_final = []
with open('../data/dataset_sft_qwen_final.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        dataset_sft_qwen_final.append(json.loads(line))

print(f"Datasets chargés ! ")

In [ ]:
dataset_dpo_qwen_final.head()

In [ ]:
dataset_sft_qwen_final

In [ ]:
import pandas as pd

# Données de test avec des PII (Personal Identifiable Information) variées
data_test = [
    {
        "prompt": "Le patient s'appelle Jean-Marc Dumont, né le 12/05/1975. Il habite au 14 rue de la Paix à Paris.",
        "chosen": "Monsieur Dumont présente une hypertension. Je lui prescris du Lisinopril.",
        "rejected": "Jean-Marc va bien, pas besoin de traitement particulier."
    },
    {
        "prompt": "Rendez-vous pour Mme Catherine Morel (06 12 34 56 78) le 22 octobre 2023 à Lyon.",
        "chosen": "L'examen de Mme Morel à Lyon montre une guérison complète.",
        "rejected": "Catherine doit revenir le 22/10/2023 pour un contrôle."
    },
    {
        "prompt": "Dossier de l'enfant Lucas Petit, habitant Marseille. Contact parent : 0491001122.",
        "chosen": "Le petit Lucas résidant à Marseille est en bonne santé.",
        "rejected": "Lucas Petit ne nécessite pas de suivi."
    }
]

df_test = pd.DataFrame(data_test)

print("✅ Dataset de test créé avec 3 exemples critiques.")

In [ ]:
test_text = "Né le 12/05/1975"
debug_results = analyzer.analyze(text=test_text, language='fr', entities=["DATE_TIME"])

for res in debug_results:
    print(f"Entité: {res.entity_type}, Score: {res.score}, Début: {res.start}, Fin: {res.end}")

In [ ]:
import re

test_text = "Le patient est né le 12/05/1975 et son tel est 06 12 34 56 78"

print("Test Date Regex:", re.findall(r"(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})", test_text))
print("Test Phone Regex:", re.findall(r"(?:(?:\+|00)33|0)\s*[1-9](?:[\s.-]*\d{2}){4}", test_text))

In [ ]:
def anonymiser_fr(text):
    if not text: return text
    
    # Analyse
    results = analyzer.analyze(text=text, entities=["DATE_TIME", "PHONE_NUMBER", "PERSON", "LOCATION"], language='fr')
    
    # Masquage
    anonymized = anonymizer.anonymize(
        text=text,
        analyzer_results=results,
        operators={
            "DATE_TIME": OperatorConfig("replace", {"new_value": "<DATE>"}),
            "PHONE_NUMBER": OperatorConfig("mask", {"chars_to_mask": 12, "masking_char": "*", "from_end": False}),
            "PERSON": OperatorConfig("replace", {"new_value": "<PATIENT>"}),
            "LOCATION": OperatorConfig("replace", {"new_value": "<LIEU>"}),
        }
    )
    return anonymized.text

In [ ]:
# On applique l'anonymisation sur le test
df_test_anonyme = df_test.copy()

for col in ['prompt', 'chosen', 'rejected']:
    df_test_anonyme[col] = df_test_anonyme[col].apply(anonymiser_texte_v4)

# Affichage pour comparaison
for i in range(len(df_test)):
    print(f"\n--- EXEMPLE {i+1} ---")
    print(f"ORIGINAL : {df_test.iloc[i]['prompt']}")
    print(f"ANONYME  : {df_test_anonyme.iloc[i]['prompt']}")

In [ ]:
data_test_en = [
    {
        "prompt": "Patient John Doe, born on 05/12/1975, lives at 742 Evergreen Terrace in Springfield.",
        "chosen": "Mr. Doe is stable. I prescribed some Aspirin.",
        "rejected": "John is doing fine, no need for further action."
    },
    {
        "prompt": "Appointment for Mrs. Jane Smith (cell: 555-0199) on October 22nd, 2023 in New York.",
        "chosen": "The exam for Jane Smith in New York shows full recovery.",
        "rejected": "Jane must return on 10/22/2023 for a check-up."
    }
]

df_test_en = pd.DataFrame(data_test_en)

In [ ]:
def anonymiser_en(text):
    if not text or not isinstance(text, str):
        return text
    
    # On change language='en'
    # Presidio possède déjà des recognizers par défaut pour 'en'
    results = analyzer.analyze(text=text, entities=["PERSON", "LOCATION", "DATE_TIME", "PHONE_NUMBER"], language='en')
    
    anonymized_result = anonymizer.anonymize(
        text=text,
        analyzer_results=results,
        operators={
            "PERSON": OperatorConfig("replace", {"new_value": "<PATIENT>"}),
            "LOCATION": OperatorConfig("replace", {"new_value": "<LOCATION>"}),
            "DATE_TIME": OperatorConfig("replace", {"new_value": "<DATE>"}),
            "PHONE_NUMBER": OperatorConfig("mask", {"chars_to_mask": 10, "masking_char": "*", "from_end": True}),
        }
    )
    return anonymized_result.text

In [ ]:
df_test_en_anonyme = df_test_en.copy()

for col in ['prompt', 'chosen', 'rejected']:
    df_test_en_anonyme[col] = df_test_en_anonyme[col].apply(anonymiser_texte_en)

# Affichage des résultats
for i in range(len(df_test_en)):
    print(f"\n--- ENGLISH EXAMPLE {i+1} ---")
    print(f"ORIGINAL : {df_test_en.iloc[i]['prompt']}")
    print(f"ANONYMOUS: {df_test_en_anonyme.iloc[i]['prompt']}")

In [ ]:
from tqdm import tqdm

def anonymiser_mixte(text):
    if not text or not isinstance(text, str):
        return text
    
    # 1. Analyse en Français (avec tes Regex FR)
    results_fr = analyzer.analyze(text=text, entities=["PERSON", "LOCATION", "DATE_TIME", "PHONE_NUMBER"], language='fr')
    
    # 2. Analyse en Anglais (avec tes Regex EN)
    results_en = analyzer.analyze(text=text, entities=["PERSON", "LOCATION", "DATE_TIME", "PHONE_NUMBER"], language='en')
    
    # 3. Fusion des résultats (Presidio gère les doublons intelligemment)
    all_results = results_fr + results_en
    
    # 4. Anonymisation unique avec les labels FR (plus simple pour la cohérence du dataset)
    anonymized = anonymizer.anonymize(
        text=text,
        analyzer_results=all_results,
        operators={
            "PERSON": OperatorConfig("replace", {"new_value": "<PATIENT>"}),
            "LOCATION": OperatorConfig("replace", {"new_value": "<LIEU>"}),
            "DATE_TIME": OperatorConfig("replace", {"new_value": "<DATE>"}),
            "PHONE_NUMBER": OperatorConfig("mask", {"chars_to_mask": 12, "masking_char": "*", "from_end": False}),
        }
    )
    return anonymized.text



In [ ]:
import pandas as pd

# On mélange des cas français, anglais et du "franglais" médical
data_test_mixte = [
    {
        "prompt": "Le patient Jean-Marc Dumont, né le 12/05/1975, was transferred to Springfield Hospital.",
        "chosen": "Patient is stable at 742 Evergreen Terrace. Monsieur Dumont va mieux.",
        "rejected": "Jean-Marc is doing fine in Paris."
    },
    {
        "prompt": "Appointment for Mrs. Jane Smith (cell: 555-0199) le 22 octobre 2023 à Lyon.",
        "chosen": "L'examen de Jane Smith à Lyon shows full recovery.",
        "rejected": "Jane must return on 10/22/2023 for a check-up."
    }
]

df_test_mixte = pd.DataFrame(data_test_mixte)
print("🧪 Dataset de test mixte créé.")

In [ ]:
from presidio_analyzer import PatternRecognizer, Pattern

# Nouvelle Regex pour les dates texte : "22 octobre 2023" ou "October 22nd, 2023"
date_text_pattern = Pattern(
    name="date_text", 
    regex=r"(\d{1,2}\s+(janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre|january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{4})", 
    score=0.8
)
date_text_rec = PatternRecognizer(supported_entity="DATE_TIME", patterns=[date_text_pattern], supported_language="fr")
analyzer.registry.add_recognizer(date_text_rec)

def anonymiser_mixte_v2(text):
    if not text or not isinstance(text, str): return text
    
    # On ajoute 'ORG' pour les hôpitaux/cliniques
    entities_to_detect = ["PERSON", "LOCATION", "DATE_TIME", "PHONE_NUMBER", "ORG"]
    
    res_fr = analyzer.analyze(text=text, entities=entities_to_detect, language='fr')
    res_en = analyzer.analyze(text=text, entities=entities_to_detect, language='en')
    
    all_results = res_fr + res_en
    
    anonymized = anonymizer.anonymize(
        text=text,
        analyzer_results=all_results,
        operators={
            "PERSON": OperatorConfig("replace", {"new_value": "<PATIENT>"}),
            "LOCATION": OperatorConfig("replace", {"new_value": "<LIEU>"}),
            "ORG": OperatorConfig("replace", {"new_value": "<HOPITAL>"}), # Protection des établissements
            "DATE_TIME": OperatorConfig("replace", {"new_value": "<DATE>"}),
            "PHONE_NUMBER": OperatorConfig("mask", {"chars_to_mask": 12, "masking_char": "*", "from_end": False}),
        }
    )
    return anonymized.text

In [ ]:
def anonymiser_mixte_ultra(text):
    if not text or not isinstance(text, str):
        return text
    
    # On ajoute 'ORG' pour capturer les noms d'hôpitaux, cliniques et labos
    entities_to_detect = ["PERSON", "LOCATION", "DATE_TIME", "PHONE_NUMBER", "ORG"]
    
    # Double analyse FR et EN
    res_fr = analyzer.analyze(text=text, entities=entities_to_detect, language='fr')
    res_en = analyzer.analyze(text=text, entities=entities_to_detect, language='en')
    
    all_results = res_fr + res_en
    
    anonymized = anonymizer.anonymize(
        text=text,
        analyzer_results=all_results,
        operators={
            "PERSON": OperatorConfig("replace", {"new_value": "<PATIENT>"}),
            "LOCATION": OperatorConfig("replace", {"new_value": "<LIEU>"}),
            "ORG": OperatorConfig("replace", {"new_value": "<ETABLISSEMENT>"}), # Pour les hôpitaux
            "DATE_TIME": OperatorConfig("replace", {"new_value": "<DATE>"}),
            "PHONE_NUMBER": OperatorConfig("mask", {"chars_to_mask": 12, "masking_char": "*", "from_end": False}),
        }
    )
    return anonymized.text

In [ ]:
# On s'assure d'avoir la version v3 qui inclut 'ORG'
def anonymiser_mixte_v3(text):
    if not text or not isinstance(text, str):
        return text
    
    # On ajoute 'ORG' pour capturer "Springfield Hospital" ou "CHU de Lyon"
    entities_to_detect = ["PERSON", "LOCATION", "DATE_TIME", "PHONE_NUMBER", "ORG"]
    
    # Double analyse FR et EN
    res_fr = analyzer.analyze(text=text, entities=entities_to_detect, language='fr')
    res_en = analyzer.analyze(text=text, entities=entities_to_detect, language='en')
    
    # On fusionne les détections
    all_results = res_fr + res_en
    
    anonymized = anonymizer.anonymize(
        text=text,
        analyzer_results=all_results,
        operators={
            "PERSON": OperatorConfig("replace", {"new_value": "<PATIENT>"}),
            "LOCATION": OperatorConfig("replace", {"new_value": "<LIEU>"}),
            "ORG": OperatorConfig("replace", {"new_value": "<ETABLISSEMENT>"}), 
            "DATE_TIME": OperatorConfig("replace", {"new_value": "<DATE>"}),
            "PHONE_NUMBER": OperatorConfig("mask", {"chars_to_mask": 12, "masking_char": "*", "from_end": False}),
        }
    )
    return anonymized.text

In [ ]:
from scripts.anonymiser import anonymiser_mixte_final

In [ ]:
# Application sur ton df de test fictif
df_test_mixte_anonyme = df_test_mixte.copy()
for col in ['prompt', 'chosen', 'rejected']:
    df_test_mixte_anonyme[col] = df_test_mixte_anonyme[col].apply(anonymiser_mixte_final)

# Vérification du résultat
print("--- RÉSULTAT DU TEST FICTIF ---")
print(df_test_mixte_anonyme['prompt'].iloc[0]) 
# Doit afficher : "... was transferred to <ETABLISSEMENT>."

In [ ]:
import pandas as pd

# On crée un dataset de test qui simule ton format DPO (prompt, chosen, rejected)
data_tests_fictifs = [
    {
        "prompt": "Le patient Jean-Marc Dumont, né le 12/05/1975, was transferred to Springfield Hospital.",
        "chosen": "Monsieur Dumont est stable au CHU de Lyon. Je prescris de l'Aspirine.",
        "rejected": "Jean-Marc is fine, call him at 06 12 34 56 78."
    },
    {
        "prompt": "Appointment for Mrs. Jane Smith (cell: 555-0199) le 22 octobre 2023 à l'Hôpital Saint-Louis.",
        "chosen": "The exam for Jane Smith in Paris shows full recovery.",
        "rejected": "Jane must return on 10/22/2023 to the Clinic for a check-up."
    },
    {
        "prompt": "Dossier de l'enfant Lucas Petit, habitant Marseille. Contact parent : 0491001122.",
        "chosen": "Le petit Lucas résidant à Marseille est en bonne santé.",
        "rejected": "Lucas Petit ne nécessite pas de suivi au Centre Médical."
    }
]

df_tests = pd.DataFrame(data_tests_fictifs)
print("✅ Dataset de test fictif généré (3 exemples complexes).")

In [ ]:
df_tests_anonyme = df_tests.copy()

for col in ['prompt', 'chosen', 'rejected']:
    df_tests_anonyme[col] = df_tests_anonyme[col].apply(anonymiser_mixte_final)

# Affichage des résultats complets
for i in range(len(df_tests_anonyme)):
    print(f"\n--- 🔎 TEST CAS {i+1} ---")
    print(f"ORIGINAL PROMPT : {df_tests.iloc[i]['prompt']}")
    print(f"ANONYME  PROMPT : {df_tests_anonyme.iloc[i]['prompt']}")
    print(f"ANONYME  CHOSEN : {df_tests_anonyme.iloc[i]['chosen']}")
    print(f"ANONYME  REJECTED: {df_tests_anonyme.iloc[i]['rejected']}")
    print("-" * 60)

# NOUVELLE ANONYMISATION

In [ ]:
import json

# Pour le DPO
dataset_dpo_qwen_final = []
with open('../data/dataset_dpo_qwen_final.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        dataset_dpo_qwen_final.append(json.loads(line))

# Pour le SFT
dataset_sft_qwen_final = []
with open('../data/dataset_sft_qwen_final.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        dataset_sft_qwen_final.append(json.loads(line))

print(f"Datasets chargés ! ")

In [ ]:
import pandas as pd

# Conversion en DataFrame si ce sont des listes ou des objets Hugging Face
if not isinstance(dataset_dpo_qwen_final, pd.DataFrame):
    print("🔄 Conversion du dataset DPO en DataFrame Pandas...")
    dataset_dpo_qwen_final = pd.DataFrame(dataset_dpo_qwen_final)

if not isinstance(dataset_sft_qwen_final, pd.DataFrame):
    print("🔄 Conversion du dataset SFT en DataFrame Pandas...")
    dataset_sft_qwen_final = pd.DataFrame(dataset_sft_qwen_final)

print("✅ Prêt pour le traitement !")

In [ ]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from tqdm import tqdm
from scripts.anonymiser import anonymiser_mixte_final
from scripts.anonymiser import anonymiser_conversation_sft


In [ ]:

print(f"🏗️ Phase 1 : Anonymisation de {len(dataset_sft_qwen_final)} conversations...")

dataset_sft_qwen_final['messages'] = [
    anonymiser_conversation_sft(m) for m in tqdm(dataset_sft_qwen_final['messages'])
]

print("🏗️ Phase 2 : Anonymisation du dataset DPO (364 lignes)...")
for col in ['prompt', 'chosen', 'rejected']:
    print(f"Traitement de la colonne {col}...")
    dataset_dpo_qwen_final[col] = [anonymiser_mixte_final(t) for t in tqdm(dataset_dpo_qwen_final[col])]

print("✨ Terminé ! Structure JSON préservée et données anonymisées.")

In [ ]:
# Sauvegarde du DPO
dataset_dpo_qwen_final.to_json(
    "../data/dataset_dpo_qwen_ANONYME_final.jsonl", 
    orient='records', 
    lines=True, 
    force_ascii=False
)

# Sauvegarde du SFT
dataset_sft_qwen_final.to_json(
    "../data/dataset_sft_qwen_ANONYME_final.jsonl", 
    orient='records', 
    lines=True, 
    force_ascii=False
)

print("💾 Fichiers sauvegardés avec succès !")

In [ ]:
import os
from datetime import datetime

# Création d'un dossier pour les exports si nécessaire
chemin = "../data/data_versioned"
os.makedirs(chemin, exist_ok=True)

version = "v1.0.0" # Première version majeure après anonymisation
date_str = datetime.now().strftime("%Y%m%d")

# Chemins de fichiers versionnés
path_dpo = f"{chemin}/chsa_dpo_bilingual_anonymized_{version}_{date_str}.jsonl"
path_sft = f"{chemin}/chsa_sft_bilingual_anonymized_{version}_{date_str}.jsonl"

# Sauvegarde finale
dataset_dpo_qwen_final.to_json(path_dpo, orient='records', lines=True, force_ascii=False)
dataset_sft_qwen_final.to_json(path_sft, orient='records', lines=True, force_ascii=False)

print(f"✅ Fichiers versionnés créés :\n- {path_dpo}\n- {path_sft}")

# NOUVEAU SPLIT

In [ ]:
dataset_dpo_qwen_final

In [ ]:
dataset_sft_qwen_final

In [ ]:
NOUVEAU_PROMPT = (
    "Tu es un médecin urgentiste et régulateur expert. Ton rôle est d'analyser "
    "rapidement des situations cliniques pour effectuer un triage précis "
    "(déterminer le niveau d'urgence et extraire les symptômes clés), ou de "
    "répondre avec rigueur et clarté aux questions médicales théoriques. "
    "Ton raisonnement et ton verdict médical final doivent être structurés "
    "sous la balise ### ANALYSE. Si tu dois poser une question pour établir "
    "un diagnostic ou répondre directement à une question médicale, "
    "utilise la balise ### ASSISTANT."
)

In [ ]:
def nettoyer_prompt_dpo(text):
    if not isinstance(text, str):
        return text
    
    # On cherche le début de la question de l'utilisateur
    if "User:" in text:
        # On garde tout ce qui commence à "User:" jusqu'à la fin
        reste_du_texte = text[text.find("User:"):]
        return f"{NOUVEAU_PROMPT}\n\n{reste_du_texte}"
    
    return text

# Application sur les colonnes du DPO
for col in ['prompt', 'chosen', 'rejected']:
    dataset_dpo_qwen_final[col] = [
        nettoyer_prompt_dpo(t) for t in tqdm(dataset_dpo_qwen_final[col])
    ]

In [ ]:
from scripts.create_triple_split import create_triple_split

# Lancement pour tes datasets anonymisés
print("🏗️ Création des versions Train/Val/Test...")
create_triple_split(dataset_sft_qwen_final, "sft")
create_triple_split(dataset_dpo_qwen_final, "dpo")

print("\n✅ Ton dataset bilingue, anonymisé et versionné est prêt pour l'entraînement !")

In [3]:
from scripts.push_to_hf import push_to_hf, push_to_hf_metadata

# Exécution
push_to_hf_metadata("sft", "../data/data_versioned", "huggingjojo/medical-bilingual-sft","chsa_sft_bilingual_anonymized_v1.0.0_20260401.jsonl")
push_to_hf("dpo", "../data/data_versioned/dpo", "huggingjojo/medical-bilingual-dpo")

🚀 Préparation de l'upload pour huggingjojo/medical-bilingual-sft...


Generating train split: 3599 examples [00:00, 136659.76 examples/s]
Generating train split: 450 examples [00:00, 59810.40 examples/s]
Generating train split: 450 examples [00:00, 55056.20 examples/s]
Generating train split: 4499 examples [00:00, 169563.14 examples/s]
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 46.22ba/s]
Processing Files (1 / 1): 100%|██████████| 5.24MB / 5.24MB, 1.87MB/s  
New Data Upload: 100%|██████████| 5.24MB / 5.24MB, 1.87MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.87s/ shards]
Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 576.22ba/s]
Processing Files (1 / 1): 100%|██████████|  651kB /  651kB,  808kB/s  
New Data Upload: 100%|██████████|  651kB /  651kB,  8

✅ huggingjojo/medical-bilingual-sft est maintenant en ligne !
🚀 Préparation de l'upload pour huggingjojo/medical-bilingual-dpo...


Generating train split: 291 examples [00:00, 40227.50 examples/s]
Generating train split: 36 examples [00:00, 5341.55 examples/s]
Generating train split: 37 examples [00:00, 6178.41 examples/s]
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 168.13ba/s]
Processing Files (1 / 1): 100%|██████████|  793kB /  793kB,  254kB/s  
New Data Upload: 100%|██████████|  793kB /  793kB,  254kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.96s/ shards]
Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 554.58ba/s]
Processing Files (1 / 1): 100%|██████████|  124kB /  124kB,  207kB/s  
New Data Upload: 100%|██████████|  124kB /  124kB,  207kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,

✅ huggingjojo/medical-bilingual-dpo est maintenant en ligne !
